Training notebook

In [1]:
import pandas as pd

In [ ]:
df=pd.read_csv('dataset/synthetic_logs.csv')

df.head()



,timestamp,source,log_message,target_label,complexity
0,2025-06-27 07:20:25,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,bert
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,bert
3,2025-07-12 00:24:16,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,bert
4,2025-06-02 18:25:23,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,bert
...,...,...,...,...,...
2405,2025-08-13 07:29:25,ModernHR,nova.osapi_compute.wsgi.server [req-96c3ec98-2...,HTTP Status,bert
2406,1/11/2025 5:32,ModernHR,User 3844 account experienced multiple failed ...,Security Alert,bert
2407,2025-08-03 03:07:47,ThirdPartyAPI,nova.metadata.wsgi.server [req-b6d4a270-accb-4...,HTTP Status,bert
2408,11/11/2025 11:52,BillingSystem,Email service affected by failed transmission,Critical Error,bert


In [7]:
df.source.unique()

array(['ModernCRM', 'AnalyticsEngine', 'ModernHR', 'BillingSystem',
       'ThirdPartyAPI', 'LegacyCRM'], dtype=object)

In [8]:
df.target_label.unique()

array(['HTTP Status', 'Critical Error', 'Security Alert', 'Error',
       'System Notification', 'Resource Usage', 'User Action',
       'Workflow Error', 'Deprecation Warning'], dtype=object)

In [33]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import DBSCAN
import numpy as np

#load pre trained sentence transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

#generate embeddings for the logs
embeddings = model.encode(df['log_message'].tolist())

#perform DBSCAN Clustering
dbscan= DBSCAN(eps=0.2, min_samples=1, metric='cosine')
clusters=dbscan.fit_predict(embeddings)

#add cluster labels to the dataframe
df['cluster']=clusters

#visualize the clusters
df['cluster']



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


0       0
1       1
2       2
3       0
4       0
       ..
2405    0
2406    7
2407    0
2408    1
2409    7
Name: cluster, Length: 2410, dtype: int64

In [37]:
df['cluster'].unique()

array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
        13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
        26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
        39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
        52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
        65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,
        78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
        91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103,
       104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116,
       117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129,
       130, 131, 132, 133, 134, 135])

In [36]:
df[df['cluster'] == 5]['log_message'].head()


8     nova.compute.claims [req-a07ac654-8e81-416d-bf...
26    nova.compute.claims [req-d6986b54-3735-4a42-90...
40    nova.compute.claims [req-72b4858f-049e-49e1-b3...
58    nova.compute.claims [req-5c8f52bd-8e3c-41f0-95...
61    nova.compute.claims [req-d38f479d-9bb9-4276-96...
Name: log_message, dtype: object

In [40]:
df[df.cluster==7]

,timestamp,source,log_message,target_label,complexity,cluster
13,8/4/2025 19:57,ThirdPartyAPI,Multiple bad login attempts detected on user 8...,Security Alert,bert,7
53,7/29/2025 0:17,ModernHR,Multiple login failures occurred on user 9052 ...,Security Alert,bert,7
72,2/3/2025 5:35,AnalyticsEngine,User 7153 made multiple incorrect login attempts,Security Alert,bert,7
74,5/9/2025 21:18,BillingSystem,User 8300 made multiple incorrect login attempts,Security Alert,bert,7
113,9/3/2025 2:46,ThirdPartyAPI,Multiple login failures were detected for user...,Security Alert,bert,7
130,12/21/2025 19:36,ThirdPartyAPI,Multiple failed login attempts were reported f...,Security Alert,bert,7
168,5/19/2025 7:56,ModernHR,Multiple incorrect login attempts were made by...,Security Alert,bert,7
173,11/30/2025 2:29,AnalyticsEngine,User 9167 experienced repeated login failures,Security Alert,bert,7
295,8/18/2025 6:57,AnalyticsEngine,Repeated failed login attempts occurred for us...,Security Alert,bert,7
318,12/28/2025 2:39,AnalyticsEngine,User 9131 made multiple failed login attempts,Security Alert,bert,7


In [49]:
#sort clusters by number of records in it. print 5 logs that belong to each cluster.
df.groupby('cluster').size().sort_values(ascending=False)

cluster
0      1017
5       147
11      100
13       86
7        60
       ... 
102       1
103       1
105       1
106       1
135       1
Length: 136, dtype: int64

In [57]:
#loop the top 5 clusters with most records and print 5 logs from each cluster
for cluster in df.groupby('cluster').size().sort_values(ascending=False).head(5).index:
    print(f"Cluster {cluster}:")
    print(df[df['cluster'] == cluster]['log_message'].head(5))
    print("\n")



Cluster 0:
0    nova.osapi_compute.wsgi.server [req-b9718cd8-f...
3    nova.osapi_compute.wsgi.server [req-4895c258-b...
4    nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...
5    nova.osapi_compute.wsgi.server [req-f0bffbc3-5...
9    nova.osapi_compute.wsgi.server [req-2bf7cfee-a...
Name: log_message, dtype: object


Cluster 5:
8     nova.compute.claims [req-a07ac654-8e81-416d-bf...
26    nova.compute.claims [req-d6986b54-3735-4a42-90...
40    nova.compute.claims [req-72b4858f-049e-49e1-b3...
58    nova.compute.claims [req-5c8f52bd-8e3c-41f0-95...
61    nova.compute.claims [req-d38f479d-9bb9-4276-96...
Name: log_message, dtype: object


Cluster 11:
27     User User685 logged out.
57      User User395 logged in.
85      User User225 logged in.
88     User User494 logged out.
126     User User900 logged in.
Name: log_message, dtype: object


Cluster 13:
30     Backup started at 2025-05-14 07:06:55.
44     Backup started at 2025-02-15 20:00:19.
108      Backup ended at 2025-08-08 13:06:

0    nova.osapi_compute.wsgi.server [req-b9718cd8-f...
3    nova.osapi_compute.wsgi.server [req-4895c258-b...
4    nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...
5    nova.osapi_compute.wsgi.server [req-f0bffbc3-5...
9    nova.osapi_compute.wsgi.server [req-2bf7cfee-a...
Name: log_message, dtype: object

In [67]:
import re
def classify_with_regex(log_message):
    regex_patterns = {
        r"User User\d+ logged (in|out).": "User Action",
        r"Backup (started|ended) at .*": "System Notification",
        r"Backup completed successfully.": "System Notification",
        r"System updated to version .*": "System Notification",
        r"File .* uploaded successfully by user .*": "System Notification",
        r"Disk cleanup completed successfully.": "System Notification",
        r"System reboot initiated by user .*": "System Notification",
        r"Account with ID .* created by .*": "User Action"
    }
    for pattern, label in regex_patterns.items():
        if re.search(pattern, log_message,re.IGNORECASE):
            return label
    return None

In [71]:
classify_with_regex('User User685 logged out...')

'User Action'

In [77]:
classify_with_regex('File uploaded successfully by user sadd')

In [79]:
classify_with_regex('Account with ID chakka created by manga')

'User Action'

In [80]:
df['regex_label']=df['log_message'].apply(classify_with_regex)

In [88]:
df[df.regex_label.notnull()]

,timestamp,source,log_message,target_label,complexity,cluster,regex_label
7,10/11/2025 8:44,ModernHR,File data_6169.csv uploaded successfully by us...,System Notification,regex,4,System Notification
14,1/4/2025 1:43,ThirdPartyAPI,File data_3847.csv uploaded successfully by us...,System Notification,regex,4,System Notification
15,5/1/2025 9:41,ModernCRM,Backup completed successfully.,System Notification,regex,8,System Notification
18,2/22/2025 17:49,ModernCRM,Account with ID 5351 created by User634.,User Action,regex,9,User Action
27,9/24/2025 19:57,ThirdPartyAPI,User User685 logged out.,User Action,regex,11,User Action
...,...,...,...,...,...,...,...
2376,6/27/2025 8:47,ModernCRM,System updated to version 2.0.5.,System Notification,regex,21,System Notification
2381,9/5/2025 6:39,ThirdPartyAPI,Disk cleanup completed successfully.,System Notification,regex,32,System Notification
2394,4/3/2025 13:13,ModernHR,Disk cleanup completed successfully.,System Notification,regex,32,System Notification
2395,5/2/2025 14:29,ThirdPartyAPI,Backup ended at 2025-05-06 11:23:16.,System Notification,regex,13,System Notification


In [94]:
df_non_regex=df[df.regex_label.isna()].copy()

In [95]:
df_non_regex.shape

(1910, 7)

In [ ]:
#in df_non_regex datframe print those target_labels that has 5 or less rows
df_non_regex['target_label'].value_counts()

target_label
HTTP Status            1017
Security Alert          371
Error                   177
Resource Usage          177
Critical Error          161
Workflow Error            4
Deprecation Warning       3
Name: count, dtype: int64

In [98]:
#filtering the ones for bert
df_non_legacy=df_non_regex[df_non_regex.source!='LegacyCRM']

df_non_legacy.source.unique()

array(['ModernCRM', 'AnalyticsEngine', 'ModernHR', 'BillingSystem',
       'ThirdPartyAPI'], dtype=object)

In [99]:
filtered_embeddings = model.encode(df_non_legacy['log_message'].tolist())

filtered_embeddings[:2]

array([[-1.02939703e-01,  3.35459337e-02, -2.20260508e-02,
         1.55098503e-03, -9.86916851e-03, -1.78956285e-01,
        -6.34410381e-02, -6.01762049e-02,  2.81108320e-02,
         5.99620081e-02, -1.72618385e-02,  1.43374118e-03,
        -1.49560049e-01,  3.15288105e-03, -5.66030778e-02,
         2.71685906e-02, -1.49890184e-02, -3.54038104e-02,
        -3.62936780e-02, -1.45410160e-02, -5.61498804e-03,
         8.75538513e-02,  4.55120578e-02,  2.50964370e-02,
         1.00187566e-02,  1.24266557e-02, -1.39923617e-01,
         7.68695921e-02,  3.14095281e-02, -4.15253639e-03,
         4.36902307e-02,  1.71250347e-02, -8.00950974e-02,
         5.74006215e-02,  1.89091638e-02,  8.55261534e-02,
         3.96398902e-02, -1.34371802e-01, -1.44367898e-03,
         3.06708645e-03,  1.76854104e-01,  4.44882922e-03,
        -1.69274919e-02,  2.24266555e-02, -4.35050316e-02,
         6.09036861e-03, -9.98167414e-03, -6.23972639e-02,
         1.07371677e-02, -6.04894478e-03, -7.14660510e-0

In [104]:
#using filtered_embeddings as X and target_label as y and trainin a logistic regression model
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Split the data into training and testing sets

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(filtered_embeddings, df_non_legacy['target_label'], test_size=0.2, random_state=42)

# Initialize the Logistic Regression model
logistic_model = LogisticRegression(max_iter=1000)

# Train the model
logistic_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = logistic_model.predict(X_test)



In [105]:
report=classification_report(y_test,y_pred)

In [106]:
print(report)

                precision    recall  f1-score   support

Critical Error       0.92      1.00      0.96        35
         Error       0.96      0.89      0.92        27
   HTTP Status       1.00      1.00      1.00       197
Resource Usage       1.00      1.00      1.00        35
Security Alert       1.00      0.99      0.99        87

      accuracy                           0.99       381
     macro avg       0.98      0.98      0.98       381
  weighted avg       0.99      0.99      0.99       381

